# TokenVector.Inference vs PyTorch — Colab T4 GPU benchmark

GPT-2 124M, seq1024, forward+backward. Mọi phép đo:
cùng VM, cùng input `[(i*37+11)%50257]`, seed 7, 2 thread, **xen kẽ A/B 3 vòng, median**.

Loss `.tkv` ≠ torch (FLOPs giống nhau; init scheme khác) — **chỉ so thời gian**.

Bench target: `.tkv` GPU (TKV_TORCH_GPU=1, ATen SDPA + LN/GELU CUDA) vs torch CUDA.


In [ ]:
!nvidia-smi | head -14


In [ ]:
# [1/5] Install .NET 10 SDK (retry 2 URLs)
import urllib.request, os
!apt-get update -qq && apt-get install -y -qq build-essential gcc libicu74 libssl3 libkrb5-3 zlib1g >/dev/null 2>&1 || apt-get install -y -qq build-essential gcc libicu-dev libkrb5-dev >/dev/null
urls = [
    'https://dot.net/v1/dotnet-install.sh',
    'https://builds.dotnet.microsoft.com/dotnet/scripts/v1/dotnet-install.sh',
]
for u in urls:
    try:
        with urllib.request.urlopen(u, timeout=30) as r:
            data = r.read()
        open('/tmp/dotnet-install.sh', 'wb').write(data)
        print('downloaded:', u, 'bytes:', len(data))
        break
    except Exception as e:
        print('FAIL', u, e)
else:
    raise RuntimeError('Tất cả URL download dotnet-install.sh đều fail')
!bash /tmp/dotnet-install.sh --channel 10.0 --install-dir /opt/dotnet
os.environ['PATH'] = '/opt/dotnet:' + os.environ['PATH']


In [ ]:
# [2/5] Clone/update repos + LD_LIBRARY_PATH (torch CUDA)
import os, glob, subprocess
print(subprocess.run('git clone --depth 1 https://github.com/nguyenhungtran18/TokenVector.git || (cd TokenVector && git pull --depth 1)', shell=True).returncode)
print(subprocess.run('git clone --depth 1 https://github.com/nguyenhungtran18/TokenVector.Inference.git || (cd TokenVector.Inference && git pull --depth 1)', shell=True).returncode)
cublas_dir = next(iter(glob.glob('/usr/local/lib/python*/dist-packages/nvidia/cublas/lib')), None)
rt_dir     = next(iter(glob.glob('/usr/local/lib/python*/dist-packages/nvidia/cuda_runtime/lib')), None)
print('cublas_dir:', cublas_dir)
print('rt_dir    :', rt_dir)
if cublas_dir:
    os.environ['LD_LIBRARY_PATH'] = cublas_dir + (':' + os.environ['LD_LIBRARY_PATH'] if os.environ.get('LD_LIBRARY_PATH') else '')
torch_dir = next(iter(glob.glob('/usr/local/lib/python*/dist-packages/torch')), None)
print('torch     :', torch_dir)
if torch_dir:
    os.environ['LD_LIBRARY_PATH'] = torch_dir + '/lib:' + os.environ.get('LD_LIBRARY_PATH', '')
print('LD_LIBRARY_PATH =', os.environ['LD_LIBRARY_PATH'])


In [ ]:
# [3/5] Build tkv_simd_native.so + tkv_step124.dll
import subprocess
print(subprocess.run('cd /content/TokenVector/Release/tools/simd_native && gcc -O2 -mavx2 -mfma -shared -fPIC tkv_simd_native.c tkv_cuda_native.c tkv_linear_dw.c tkv_bwd_ops.c tkv_flash_attn.c tkv_step124.c tkv_prof.c tkv_attn_cuda.c -lm -lpthread -ldl -o tkv_simd_native.dll', shell=True).returncode)
print(subprocess.run('cp /content/TokenVector/Release/tools/simd_native/tkv_simd_native.dll /content/TokenVector.Inference/demo/tkv_simd_native.dll && cp /content/TokenVector/Release/tools/simd_native/tkv_simd_native.dll /content/TokenVector.Inference/tkv_simd_native.dll', shell=True).returncode)
print(subprocess.run('cd /content/TokenVector/Release/tools/simd_native && gcc -O2 -mavx2 -mfma -shared -fPIC tkv_step124.c -o tkv_step124.dll', shell=True).returncode)
print(subprocess.run('cp /content/TokenVector/Release/tools/simd_native/tkv_step124.dll /content/TokenVector.Inference/demo/tkv_step124.dll', shell=True).returncode)
import ctypes
lib = ctypes.CDLL('/content/TokenVector.Inference/demo/tkv_simd_native.dll')
lib.TkvCUDA_Available.restype = ctypes.c_int
print('TkvCUDA_Available() =', lib.TkvCUDA_Available(), '(1=cuBLAS, 0=CPU)')


In [ ]:
# [4/5] Build tkv_torch_fused.so (ATen bridge)
import subprocess, os, sys
torch_dir = subprocess.check_output([sys.executable, '-c', 'import torch,os; print(os.path.dirname(torch.__file__))']).decode().strip()
print('torch:', torch_dir)
inc1 = torch_dir + '/include'
inc2 = torch_dir + '/include/torch/csrc/api/include'
libdir = torch_dir + '/lib'
cmd = ('cd /content/TokenVector/Release/tools/simd_native && '
       + 'g++ -O2 -shared -std=c++20 -fPIC torch_fused.cpp -o tkv_torch_fused.so '
       + '-I' + inc1 + ' -I' + inc2 + ' -L' + libdir + ' '
       + '-Wl,-rpath,' + libdir + ' '
       + '-ltorch -ltorch_cpu -lc10 -lc10_cuda -ltorch_cuda')
p = subprocess.run(cmd, shell=True, capture_output=True, text=True, timeout=1800)
print('build return code:', p.returncode)
if p.returncode != 0:
    print('STDERR:
', p.stderr[-4000:])
print('build tail:
', (p.stdout or '')[-1500:])
so_path = '/content/TokenVector/Release/tools/simd_native/tkv_torch_fused.so'
print('exists:', os.path.exists(so_path))
os.environ['TKV_TORCH_PATH'] = so_path
os.environ['LD_LIBRARY_PATH'] = libdir + ':' + os.environ.get('LD_LIBRARY_PATH', '')


In [ ]:
# [5/5] Benchmark: .tkv GPU (ATen bridge) vs torch CUDA — xen ke x3, median
import subprocess, re, statistics as st
DOT = '/opt/dotnet/dotnet'
def run(cmd):
    p = subprocess.run(cmd, shell=True, capture_output=True, text=True, timeout=1800)
    return (p.stdout or '') + '\n' + (p.stderr or '')
def tkv_gpu():
    out = run('cd TokenVector.Inference && printf "1024\n2\n1\n" > tv/tests/_seq_cfg.txt && TKV_TORCH_GPU=1 ' + DOT + ' demo/bseq.dll')
    m = re.search(r'fwd=([\d.]+) loss=([\d.]+) bwd=([\d.]+) opt=([\d.]+) copyback=([\d.]+) TONG=([\d.]+)', out)
    if not m:
        print('RAW:\n', out)
        raise RuntimeError('no BPHASE line')
    return dict(zip(('fwd', 'ce', 'bwd', 'opt', 'copyback', 'total'), map(float, m.groups())))
def torch_cuda():
    out = run('cd TokenVector.Inference && TKV_BENCH_DEVICE=cuda python3 benchmark/bench_torch.py d 1024')
    fw = re.search(r'fwd\+bwd seq1024 dev=cuda: median=([\d.]+)', out)
    fu = re.search(r'full step seq1024 dev=cuda: median=([\d.]+)', out)
    return dict(fwdbwd=float(fw.group(1)), full=float(fu.group(1)))
print('Interleaved 3 rounds / side..')
tk = []; tt = []
for r in range(3):
    if r % 2 == 0:
        a, b = tkv_gpu(), torch_cuda()
    else:
        b, a = torch_cuda(), tkv_gpu()
    tk.append(a); tt.append(b)
    print('%d: .tkv fwd=%.0f CE=%.0f bwd=%.0f opt=%.0f TONG=%.0f | torch fwd+bwd=%.1f full=%.1f' % (r+1, a['fwd'], a['ce'], a['bwd'], a['opt'], a['total'], b['fwdbwd'], b['full']))
med = st.median
fb  = med([x['fwd']+x['ce']+x['bwd'] for x in tk])
tb  = med([x['total'] for x in tk])
tfb = med([x['fwdbwd'] for x in tt])
ttf = med([x['full'] for x in tt])
print()
print('--- MEDIAN (n=3, xen ke) ---')
print('.tkv fwd+bwd   : %.1f ms | torch fwd+bwd   : %.1f ms | %.2fx' % (fb, tfb, fb/tfb))
print('.tkv full step : %.1f ms | torch full step : %.1f ms | %.2fx' % (tb, ttf, tb/ttf))


In [ ]:
# [6/5] Op breakdown TKV_PROF=1 (diagnostic)
DOT = '/opt/dotnet/dotnet'
def run(cmd):
    p = subprocess.run(cmd, shell=True, capture_output=True, text=True, timeout=1800)
    print((p.stdout or '') + (p.stderr or ''))
print('=== native (TKV_TORCH_PATH=) ===')
run('cd TokenVector.Inference && printf "1024\n2\n1\n" > tv/tests/_seq_cfg.txt && TKV_TORCH_PATH= TKV_PROF=1 ' + DOT + ' demo/bseq.dll')
print('=== GPU bridge (TKV_TORCH_GPU=1) ===')
run('cd TokenVector.Inference && printf "1024\n2\n1\n" > tv/tests/_seq_cfg.txt && TKV_TORCH_GPU=1 TKV_PROF=1 ' + DOT + ' demo/bseq.dll')


In [ ]:
# [7/5] Diagnostic — ATen attention backward có chạy hay không?
DOT = '/opt/dotnet/dotnet'
import subprocess
p = subprocess.run('cd TokenVector.Inference && printf "1024\n2\n1\n" > tv/tests/_seq_cfg.txt && TKV_TORCH_GPU=1 TKV_PROF=1 ' + DOT + ' demo/bseq.dll', shell=True, capture_output=True, text=True, timeout=1800)
out = (p.stdout or '') + '\n' + (p.stderr or '')
for line in out.splitlines():
    if any(k in line for k in ('BPHASE', '[tkv-attn-bwd]', 'FAIL')):
        print(line.rstrip())


In [ ]:
# [8/5] Peak RAM (.tkv mode 0, native)
import subprocess
print(subprocess.run('cd TokenVector.Inference && printf "1024\n0\n1\n" > tv/tests/_seq_cfg.txt && TKV_TORCH_PATH= /usr/bin/time -v /opt/dotnet/dotnet demo/bseq.dll 2>&1 | grep -E "BSEQ|Maximum resident"', shell=True, capture_output=True, text=True).stdout)
